# 4.4 Persistent 持久化核

## 概述

上一节已经实现核内多个输出块的流水。本节继续研究这些输出任务如何由有限的核反复处理：先理解 Persistent 的任务遍历原理，再将基础版的连续区间循环改写为静态跨步遍历。输入、矩阵乘主体和流水配置保持不变。

## 学习目标

- 理解固定数量的核反复处理输出任务的方式。
- 根据任务编号推导 Persistent 的静态跨步序列。
- 实现 Persistent GEMM，并验证任务覆盖与输出正确性。

[本章导航](README.md) · [上一节](04.03_pipeline.ipynb) · [下一节](04.05_practice.ipynb)

<p><small>执行前完成第1.4节环境准备，按本章README的统一入口运行。每页使用新内核；NPU程序由独立子进程执行，notebook内核不初始化设备。</small></p>

## 1. Persistent 任务遍历原理

### T.Pipelined 与 T.Persistent 分别负责什么？

上一节的 `T.Pipelined` 表达核内循环及其流水配置：它产生本核的循环编号 `it`，但不自动决定本核负责哪些全局输出块。连续分配来自我们手写的 `idx=bx×144+it`，并不是 `T.Pipelined` 只能连续分配。

`T.Persistent` 接收全局任务范围、参与核数和本核编号，直接产生分配给本核的任务编号 `idx`。本例的一维任务范围按 `idx=bx+it×8` 静态跨步遍历；它也接受 `num_stages`，可以继续表达核内流水。因此，本节改变的是任务编号生成方式，同时保留上一节的流水配置。

<table style="margin-left: 0; margin-right: auto; text-align: left;">
<thead>
<tr>
<th style="text-align: left; vertical-align: top;">对比项</th>
<th style="text-align: left; vertical-align: top;">T.Pipelined</th>
<th style="text-align: left; vertical-align: top;">T.Persistent（本例一维任务范围）</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left; vertical-align: top;">循环输入</td>
<td style="text-align: left; vertical-align: top;">本核循环范围 <code>tasks//8</code></td>
<td style="text-align: left; vertical-align: top;">全局任务范围 <code>[tasks]</code>、核数 <code>8</code>、本核编号 <code>bx</code></td>
</tr>
<tr>
<td style="text-align: left; vertical-align: top;">循环变量</td>
<td style="text-align: left; vertical-align: top;">本核循环编号 <code>it</code></td>
<td style="text-align: left; vertical-align: top;">全局输出任务编号 <code>idx</code></td>
</tr>
<tr>
<td style="text-align: left; vertical-align: top;">核间任务分配</td>
<td style="text-align: left; vertical-align: top;">由循环体中的索引公式决定</td>
<td style="text-align: left; vertical-align: top;">按指定任务范围和核编号生成静态跨步序列</td>
</tr>
<tr>
<td style="text-align: left; vertical-align: top;">流水配置</td>
<td style="text-align: left; vertical-align: top;">通过 <code>num_stages</code> 指定</td>
<td style="text-align: left; vertical-align: top;">同样通过 <code>num_stages</code> 指定</td>
</tr>
</tbody>
</table>

据此，将“`T.Pipelined` 加手写任务索引”替换为 `T.Persistent`，是把任务遍历交给专门接口表达；不是用任务调度代替流水，也不代表只有 Persistent 才能让一个核处理多块数据。

### 从已有核内循环出发

本例有 1152 个输出 tile，8 个核任务各处理 144 个。上一节将连续区间分给各核：core bx 的第 it 项为 `idx=bx×144+it`。这已经具备一个核连续处理多个任务的特征。

Persistent 将“由有限的核遍历更多输出任务”直接表达为循环。在当前后端，本例的 `T.Persistent([tasks],8,bx)` 按 `idx=bx+it×8` 静态跨步：core0 处理 0、8、16…1144；core1 处理 1、9、17…1145。**它不是空闲核动态抢任务。**

![8 核的两种任务遍历方式](images/04.04_eight_persistent_v2.svg)

`idx` 是输出 tile 编号，不是元素索引。由 `mi=idx//N_tiles`、`ni=idx%N_tiles` 还原二维输出位置。验证全部编号恰好出现一次，才能保证任务不重复、不遗漏。

In [1]:
tasks = 1152
blocks = 8
contiguous = {bx: list(range(bx * (tasks // blocks), (bx + 1) * (tasks // blocks))) for bx in range(blocks)}
persistent = {bx: list(range(bx, tasks, blocks)) for bx in range(blocks)}
for label, mapping in [('contiguous', contiguous), ('persistent', persistent)]:
    ids = [i for seq in mapping.values() for i in seq]
    assert sorted(ids) == list(range(tasks)) and len(set(ids)) == tasks
    print(label, 'full coverage PASS; cores=', len(mapping))
    for bx, seq in mapping.items():
        print('core', bx, 'first=', seq[:4], 'last=', seq[-1], 'count=', len(seq))

contiguous full coverage PASS; cores= 8
core 0 first= [0, 1, 2, 3] last= 143 count= 144
core 1 first= [144, 145, 146, 147] last= 287 count= 144
core 2 first= [288, 289, 290, 291] last= 431 count= 144
core 3 first= [432, 433, 434, 435] last= 575 count= 144
core 4 first= [576, 577, 578, 579] last= 719 count= 144
core 5 first= [720, 721, 722, 723] last= 863 count= 144
core 6 first= [864, 865, 866, 867] last= 1007 count= 144
core 7 first= [1008, 1009, 1010, 1011] last= 1151 count= 144
persistent full coverage PASS; cores= 8
core 0 first= [0, 8, 16, 24] last= 1144 count= 144
core 1 first= [1, 9, 17, 25] last= 1145 count= 144
core 2 first= [2, 10, 18, 26] last= 1146 count= 144
core 3 first= [3, 11, 19, 27] last= 1147 count= 144
core 4 first= [4, 12, 20, 28] last= 1148 count= 144
core 5 first= [5, 13, 21, 29] last= 1149 count= 144
core 6 first= [6, 14, 22, 30] last= 1150 count= 144
core 7 first= [7, 15, 23, 31] last= 1151 count= 144


## 2. Persistent GEMM 实现

### 从连续区间改为跨步遍历

保持 T.Kernel、输入输出类型、L1/L0C 分配和计算主体不变。将 `T.Pipelined(tasks//8,...)` 与 `idx=bx×(tasks//8)+it`，替换成 `T.Persistent([tasks],8,bx,num_stages=stages)`。相同的 stages 保留上一节的流水配置。

以下实现仍计算完整 K 的输出块，因此每次 `T.gemm` 都从新的结果开始。

![从任务编号到输出区域](images/04.04_task_to_output_v2.svg)

例如 core1 第 2 轮（it=1）得到 idx=9，映射为 mi=0、ni=9，处理 C[0:128,1152:1280]。每完成一个输出块，就按步长 8 领取本核的下一个编号。

```python
import tilelang
import tilelang.language as T


def persistent_gemm(M=512, N=36864, K=256, BM=128, BN=128, stages=2):
    tasks = (M // BM) * (N // BN)

    @T.prim_func
    def main(
        A: T.Tensor((M, K), 'bfloat16'),
        B: T.Tensor((N, K), 'bfloat16'),
        C: T.Tensor((M, N), 'float32'),
    ):
        with T.Kernel(8) as bx:
            a = T.alloc_l1((BM, K), 'bfloat16')
            b = T.alloc_l1((BN, K), 'bfloat16')
            c = T.alloc_l0c((BM, BN), 'float32')
            T.annotate_buffer_versions({a: stages, b: stages, c: stages})

            for idx in T.Persistent([tasks], 8, bx, num_stages=stages):
                mi = idx // (N // BN)
                ni = idx % (N // BN)
                T.copy(A[mi * BM : (mi + 1) * BM, :], a)
                T.copy(B[ni * BN : (ni + 1) * BN, :], b)
                T.gemm(a, b, c, transpose_B=True, clear_accum=True)

                T.copy(c, C[mi * BM : (mi + 1) * BM, ni * BN : (ni + 1) * BN])

    return main
```

[共用可执行工厂](src/gemm_kernel.py)通过 `persistent=True` 选择该路径。用 `(512,36864,256)`、BM=BN=128、stages=2 验证全部输出；随机、零、常数和周期输入各调用两次，阈值保持 `rtol=atol=1e-4`。

In [2]:
import subprocess, sys
p = subprocess.run([sys.executable, 'src/run_gemm.py', 'persistent'], text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
print("\n".join(line for line in p.stdout.splitlines()
                       if not line.startswith("Loading tilelang libs from dev root:")))
p.check_returncode()

[W917 20:46:04.097480916 FunctionLoader.cpp:48] Warning: LD_PRELOAD detected, FunctionLoader prefers RTLD_DEFAULT for symbol resolution. (function operator())
2026-09-17 20:46:11  [TileLang:tilelang.jit.kernel:INFO] (kernel.py:131): TileLang begins to compile kernel `main` with `out_idx=[2]`
2026-09-17 20:46:22  [TileLang:tilelang.jit.kernel:INFO] (kernel.py:139): TileLang completes to compile kernel `main`
512_36864_256_128_128_s2_p1 random: full-output PASS; shape=(512, 36864); max_abs_error=2.3841858e-06
512_36864_256_128_128_s2_p1 zero: full-output PASS; shape=(512, 36864); max_abs_error=0
512_36864_256_128_128_s2_p1 constant: full-output PASS; shape=(512, 36864); max_abs_error=0
512_36864_256_128_128_s2_p1 pattern: full-output PASS; shape=(512, 36864); max_abs_error=0
Verification passed!



## 3. 任务遍历与二维映射

Persistent 确定每个核遍历哪些任务编号，编号到二维输出块的映射还可以独立设计。swizzle 就是一类遍历顺序重排，例如优先访问可复用同一 A 或 B 输入块的输出任务。本节采用简单的行优先映射，先掌握任务编号和二维位置的关系。进一步阅读见[原始示例](src/original/example_gemm.py)。

![行优先与分组遍历的示意](images/04.04_swizzle.svg)

图中用 2×4 的小网格说明重排：两种顺序覆盖相同输出块，但相邻任务可能复用的输入不同。正文实现仍采用行优先映射，不额外改变算子。

## 小结

Persistent 用循环表达固定数量的核反复处理输出任务。本节推导了静态跨步编号，将其接入相同 GEMM 主体，并验证了完整任务覆盖与输出结果。它也可用于 Vector 算子，使用方法不局限于矩阵乘。

## 课后练习

1. N_tiles=288 时，写出 Persistent 中 core0、core1 前四项的 `(mi,ni)`。
2. 说明 tasks、bx、idx、mi/ni 分别代表什么；为什么不能直接用 idx 作为 GM 元素偏移？

In [3]:
from pathlib import Path
print(Path("answer/04.04_answer.txt").read_text())

1. N_tiles=288：core0前四项idx为0、8、16、24，对应(0,0)、(0,8)、(0,16)、(0,24)；core1为(0,1)、(0,9)、(0,17)、(0,25)。
2. tasks是输出tile总数，bx是核任务编号，idx是输出tile编号，mi/ni是二维tile坐标；GM元素范围还需要分别乘以BM/BN。

